In [ ]:
import sys
from pathlib import Path
import os
import numpy as np

# Get the directory where this notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent  # Go up one level to project root
src_path = project_root / 'src'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cs_priors.plotting.plotting import (
    plot_equation,
    plot_two_line_equation,
    plot_overview,
    wrapper_plot_geometry,
)
from cs_priors.simulation.mixing_model import run_simulation
import  cs_priors.solvers.sparse_prior as sp
import cs_priors.solvers.vectorized_sparse_prior as vsp

In [ ]:
num_sources = 10


sim = run_simulation(num_mics=8, num_sources=num_sources, s_sparse=2, angle_step=2*np.pi/num_sources)
wrapper_plot_geometry(sim, figsize=(16,16), fontsize=16, skip_labels=True)
Y = sim.Y[:,1]
A = sim.C[:,:,1]
X0 = np.linalg.pinv(A) @ Y
X_TRUE = sim.X[:,1]

In [ ]:
plot_equation(A, Y, X0, ("A", "Y", "X0"))
print(A.shape, A)

In [ ]:
print("A has shape", A.shape)

X_sp, B = sp.sparse_prior_solution(X0, A)  # Correct: pass X0 first, then A
X_vsp, B = vsp.sparse_prior_solution(X0, A)

In [ ]:
import time
import scipy.optimize as optimize

def covariance_matrices(
    num_sources: int, variance: float = 1.0, spread: float = 0.005
) -> list[np.ndarray]:
    """Returns diagonal elements of each covariance matrix"""
    return [
        np.array([variance if j == i else spread for j in range(num_sources)])
        for i in range(num_sources)
    ]

def precompute_hessian_constant(B_real, D_diags):
    """Precompute B.T @ D_i @ B for all diagonal matrices D_i"""
    return [B_real.T @ np.diag(d) @ B_real for d in D_diags]

def first_derivative_objective(z: np.ndarray) -> np.ndarray:
    # (-1) e^{-(x_0 + Bz)^T D (x_0+Bz)}(B^T(D + D^T) (x_0 + B z))
    # where D_i is symmetric
    x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
    x_flat = x.flatten()
    grad = np.zeros_like(z)

    
    for d_i in D_diags:
        # Diagonal matrix multiply: D @ x = d * x (element-wise)
        qf = np.dot(x_flat * d_i, x_flat)  # x.T @ D @ x where D is diagonal
        exp_neg_qf = np.exp(-qf)
        # D @ x as element-wise multiply, then B.T @ (D @ x)
        grad += 2 * exp_neg_qf * (B_real.T @ (d_i * x_flat))
    return grad

def hessian_vector_product(z: np.ndarray, v: np.ndarray) -> np.ndarray:
    """Compute Hessian @ v without forming the full Hessian matrix"""
    x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
    x_flat = x.flatten()
    hess_v = np.zeros_like(v)
    
    for i, d_i in enumerate(D_diags):
        qf = np.dot(x_flat * d_i, x_flat)
        exp_neg_qf = np.exp(-qf)
        
        # B.T @ (D @ x) where D is diagonal
        BdX = B_real.T @ (d_i * x_flat)
        
        # (outer(BdX, BdX) @ v) = BdX * (BdX @ v) - avoids forming outer product!
        BdX_dot_v = np.dot(BdX, v)
        
        # Hessian-vector product: [2*outer(BdX, BdX) - 2*BtDB] @ v
        hess_v += exp_neg_qf * (2 * BdX * BdX_dot_v - 2 * (BtDB_precomputed[i] @ v))
    
    return -hess_v  # negative Hessian

def optimize_objective(X0_real, B_real, D_diags, callback=None, z_start=None, method='l-BFGS-B'):
    def negative_objective(z: np.ndarray) -> float:
        x = X0_real.reshape(-1, 1) + (B_real @ z.reshape(-1, 1))
        x_flat = x.flatten()
        return -sum(np.exp(-np.dot(x_flat * d_i, x_flat)) for d_i in D_diags)

    if z_start is None:
        z_start = np.zeros(B_real.shape[1])

    start_time = time.time()
    res = optimize.minimize(
        negative_objective,
        z_start,
        method=method,
        callback=callback,
        jac=first_derivative_objective,
        hessp=hessian_vector_product,  # Use hessp instead of hess
    )
    end_time = time.time()
    print(f"Optimization time ({method}): {end_time - start_time:.4f} seconds")
    
    z_opt = res.x
    x_opt = X0_real.reshape(-1, 1) + (B_real @ z_opt.reshape(-1, 1))
    x_opt_complex = from_real_augmented(x_opt)
    return z_opt, x_opt, x_opt_complex, res

U, S, Vt = np.linalg.svd(A)
rank = np.sum(S > 1e-10)

if rank == A.shape[1]:
    print("No null space detected.")

B = Vt[rank:].T
B_real = np.block([[B.real, -B.imag], [B.imag, B.real]])
X0_real = to_real_augmented(X0)
D_diags = covariance_matrices(num_sources=X0_real.shape[0])

# Precompute constant Hessian terms
BtDB_precomputed = precompute_hessian_constant(B_real, D_diags)

z_opt, x_opt, x_opt_complex, res = optimize_objective(X0_real, B_real, D_diags, method='l-BFGS-B')

In [ ]:
X_sp, B = sparse_prior_solution(X0, A)